# Cartera eficiente (Markowitz) - Análisis de Carteras

Este notebook:
1. Lee el archivo `empresas_byma.xlsx` directamente desde el repositorio de GitHub.
2. Toma solo las empresas marcadas con **x** en la columna `Analizar (x)`.
3. Descarga precios históricos de esas acciones (Yahoo Finance, vía `yfinance`).
4. Calcula la **frontera eficiente de Markowitz**, la **cartera de mínima varianza** y la **cartera de máximo Sharpe ratio**.

**Cómo usarlo:** `Entorno de ejecución -> Ejecutar todas` (Runtime -> Run all). No hace falta tocar nada salvo, opcionalmente, los parámetros de la celda de configuración.

**Importante:** los precios están en pesos argentinos (ARS) nominales, sin ajustar por inflación. Esto no es asesoramiento financiero, es una herramienta de análisis exploratorio.

In [ ]:
!pip install -q yfinance openpyxl

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.optimize import minimize

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## Configuración

In [ ]:
# URL raw del Excel en GitHub (rama main). Si cambiás el nombre del archivo o la rama, actualizá esta URL.
EXCEL_URL = "https://raw.githubusercontent.com/alanartola/analisis_carteras/main/empresas_byma.xlsx"

# Ventana histórica usada para estimar retornos y riesgo
LOOKBACK_PERIOD = "3y"   # ej: "1y", "2y", "3y", "5y"

# Tasa libre de riesgo anual (en la misma moneda/base que los retornos, ARS nominal). Ajustala si querés.
RISK_FREE_RATE = 0.0

# Cantidad de carteras aleatorias para dibujar la nube de la frontera eficiente
N_PORTFOLIOS = 20000

# Sin ventas en corto: pesos entre 0% y 100% por activo
ALLOW_SHORT = False

## 1. Cargar el Excel y filtrar las empresas marcadas con x

In [ ]:
universo = pd.read_excel(EXCEL_URL, sheet_name="Universo BYMA")

marca = universo["Analizar (x)"].astype(str).str.strip().str.lower()
seleccion = universo[marca.isin(["x", "si", "sí"])].copy()

if seleccion.empty:
    raise ValueError(
        "No hay ninguna empresa marcada con 'x' en la columna 'Analizar (x)' del Excel. "
        "Marcá al menos 2 empresas, subí el cambio a GitHub y volvé a ejecutar el notebook."
    )

print(f"Empresas seleccionadas: {len(seleccion)}")
seleccion[["Ticker", "Empresa", "Sector", "Ticker Yahoo Finance"]]

## 2. Descargar precios históricos

In [ ]:
tickers = seleccion["Ticker Yahoo Finance"].tolist()
nombres = dict(zip(seleccion["Ticker Yahoo Finance"], seleccion["Ticker"]))

raw = yf.download(tickers, period=LOOKBACK_PERIOD, auto_adjust=True, progress=False)
precios = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]].rename(columns={"Close": tickers[0]})

# Descarta columnas sin datos suficientes (ticker sin cotización, delisted, etc.)
precios = precios.dropna(axis=1, thresh=int(len(precios) * 0.7))
faltantes = set(tickers) - set(precios.columns)
if faltantes:
    print(f"Aviso: sin datos suficientes para {sorted(faltantes)}; se excluyen del análisis.")

precios = precios.rename(columns=nombres).dropna()

if precios.shape[1] < 2:
    raise ValueError("Se necesitan al menos 2 acciones con datos válidos para calcular una cartera. Marcá más empresas en el Excel.")

precios.tail()

## 3. Retornos, riesgo y matriz de covarianza (anualizados)

In [ ]:
retornos_diarios = precios.pct_change().dropna()

DIAS_HABILES = 252
retornos_anuales = retornos_diarios.mean() * DIAS_HABILES
cov_anual = retornos_diarios.cov() * DIAS_HABILES

resumen = pd.DataFrame({
    "Retorno anual esperado": retornos_anuales,
    "Volatilidad anual": np.sqrt(np.diag(cov_anual)),
})
resumen

## 4. Optimización de Markowitz: mínima varianza y máximo Sharpe

In [ ]:
activos = list(precios.columns)
n = len(activos)
mu = retornos_anuales.values
cov = cov_anual.values


def rendimiento_cartera(w):
    return float(np.dot(w, mu))


def volatilidad_cartera(w):
    return float(np.sqrt(np.dot(w, np.dot(cov, w))))


def sharpe_negativo(w):
    vol = volatilidad_cartera(w)
    if vol == 0:
        return 0.0
    return -(rendimiento_cartera(w) - RISK_FREE_RATE) / vol


bounds = tuple((-1.0, 1.0) if ALLOW_SHORT else (0.0, 1.0) for _ in range(n))
restricciones = ({"type": "eq", "fun": lambda w: np.sum(w) - 1.0},)
w0 = np.repeat(1.0 / n, n)

opt_min_var = minimize(volatilidad_cartera, w0, method="SLSQP", bounds=bounds, constraints=restricciones)
opt_max_sharpe = minimize(sharpe_negativo, w0, method="SLSQP", bounds=bounds, constraints=restricciones)

pesos_min_var = opt_min_var.x
pesos_max_sharpe = opt_max_sharpe.x

print("Optimización mínima varianza:", "OK" if opt_min_var.success else opt_min_var.message)
print("Optimización máximo Sharpe:  ", "OK" if opt_max_sharpe.success else opt_max_sharpe.message)

## 5. Frontera eficiente (nube de carteras simuladas)

In [ ]:
rng = np.random.default_rng(42)
resultados = np.zeros((N_PORTFOLIOS, 3))

for i in range(N_PORTFOLIOS):
    if ALLOW_SHORT:
        w = rng.normal(size=n)
    else:
        w = rng.random(n)
    w = w / np.sum(w)
    ret = rendimiento_cartera(w)
    vol = volatilidad_cartera(w)
    sharpe = (ret - RISK_FREE_RATE) / vol if vol > 0 else 0.0
    resultados[i] = [vol, ret, sharpe]

plt.figure(figsize=(10, 7))
sc = plt.scatter(resultados[:, 0], resultados[:, 1], c=resultados[:, 2], cmap="viridis", s=6, alpha=0.5)
plt.colorbar(sc, label="Sharpe ratio")

plt.scatter(volatilidad_cartera(pesos_min_var), rendimiento_cartera(pesos_min_var),
            marker="*", color="red", s=400, label="Mínima varianza")
plt.scatter(volatilidad_cartera(pesos_max_sharpe), rendimiento_cartera(pesos_max_sharpe),
            marker="*", color="gold", edgecolor="black", s=400, label="Máximo Sharpe")

plt.xlabel("Volatilidad anual (desvío estándar)")
plt.ylabel("Retorno anual esperado")
plt.title("Frontera eficiente de Markowitz")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Carteras óptimas: composición y métricas

In [ ]:
tabla_pesos = pd.DataFrame({
    "Mínima varianza": pesos_min_var,
    "Máximo Sharpe": pesos_max_sharpe,
}, index=activos)

tabla_pesos_pct = (tabla_pesos * 100).round(2)
print("Composición de las carteras óptimas (%):")
display(tabla_pesos_pct)

metricas = pd.DataFrame({
    "Retorno anual esperado": [rendimiento_cartera(pesos_min_var), rendimiento_cartera(pesos_max_sharpe)],
    "Volatilidad anual": [volatilidad_cartera(pesos_min_var), volatilidad_cartera(pesos_max_sharpe)],
    "Sharpe ratio": [
        (rendimiento_cartera(pesos_min_var) - RISK_FREE_RATE) / volatilidad_cartera(pesos_min_var),
        (rendimiento_cartera(pesos_max_sharpe) - RISK_FREE_RATE) / volatilidad_cartera(pesos_max_sharpe),
    ],
}, index=["Mínima varianza", "Máximo Sharpe"])
print("\nMétricas de las carteras óptimas:")
display(metricas)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col in zip(axes, tabla_pesos_pct.columns):
    pesos_no_cero = tabla_pesos_pct[col][tabla_pesos_pct[col] > 0.5]
    ax.pie(pesos_no_cero, labels=pesos_no_cero.index, autopct="%1.1f%%", startangle=90)
    ax.set_title(col)
plt.suptitle("Composición de la cartera óptima")
plt.show()

## Notas y limitaciones

- Los retornos se calculan sobre precios **nominales en pesos argentinos**, sin ajustar por inflación ni dividendos más allá de lo que `yfinance` ajusta automáticamente.
- El modelo de Markowitz asume que el futuro se parece al pasado (usa media y covarianza históricas); no es una predicción.
- Por defecto no se permiten ventas en corto (`ALLOW_SHORT = False`); podés cambiarlo en la celda de configuración.
- `RISK_FREE_RATE` es un supuesto que vos definís; ajustalo a una tasa libre de riesgo razonable en pesos si querés un Sharpe ratio más representativo.
- Esto es una herramienta de análisis exploratorio, no una recomendación de inversión.